# Paper replication


### **Data needed**
1. Daily prices for all NYSE/AMEX listed securities for 1974-1986 and 2012-2024, with market cap
2. Quarterly Earnings Per Share - Excludig Extraordinary Items, Price Close - Quarter End, Earnings Announcement Data (for 1982-1987 and 2020-2025)



### **Data sources**

Wharton Research Data Services. "WRDS" wrds.wharton.upenn.edu, accessed 2026-06-04.

##### **CRSP Daily**

*Query parameters*
- (primaryexch) Primary Exchange (A:AMEX, B:BATS, I:IEX, N:NYSE, Q:NASDAQ, R:NYSE ARCA, X:Unknown)
- (securitynm) Security Name
- (securitytype) Security Type
- (permco) PERMCO
- (ticker) Ticker
- (dlyprc) Daily Price
- (dlycap) Daily Capitalization
- (dlyopen) Daily Open
- (dlyclose) Daily Close
- (dlylow) Daily Low
- (dlyhigh) Daily High

*Files*
- ```CRSP_Daily_1974_1986.csv``` (2.09GB)         From 1974-01-01 to 1986-12-31
- ``CRSP_Daily_2012_2025.csv`` (469.0MB)                  From 2012-01-01 to 2024-12-31


##### **Compustat Quarterly**

*Query parameters*
- (conm) Company Name
- (tic) Ticker Symbol
- (permco) PERMCO
- primaryexch) Primary Exchange
- (rda) Report Date of Quarterly Earnings
- (fyearq) Fiscal Year
- (fqtr) Fiscal Quarter
- (proca) Price Close - Quarter
- (epsfxq) Earnings Per Share (Diluted) - Excluding Extraordinary items
- (epspxq) Earnings Per Share (Basic) - Excluding Extraordinary Items
- (shoq) Common Shares Outstanding
- (cshprq) Common Shares Used to Calculate Earnings Per Share - Basic
- (ajexq) Adjustment Factor (Company) - Cumulative by Ex-Date

*Files*
- ``Compustat_Q_196103_198712.csv`` (48.2MB)    From 1961-03 to 1987-12
- ``Compustat_Q_199903_202512.csv`` (137.2MB)   From 1999-03 to 2025-12

We first load Compustat Quarterly dataset obtained from WRDS. We only keep companies that:
- are present in the 1982-1987 window
- are listed on the NYSE/AMEX exchanges
- have at least ten consecutive earnings observations
We use epspxq (basic EPS) instead of epsfxq (diluted EPS) since more firms are shortlisted using the former.


In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

dfQ = pd.read_csv("data/Compustat_Q_196103_198712.csv")
dfD_oos = pd.read_csv("data/Compustat_Q_199903_202512.csv")

# We select firms present in the 1982-1987 window
dfQ['datadate'] = pd.to_datetime(dfQ['datadate'])
dfQ_slice = dfQ[(dfQ['datadate'] >= '1982-01-01') & (dfQ['datadate'] <= '1987-12-31')]
tickers_1982_1987 = dfQ_slice['tic'].dropna().unique()
print(f"Number of unique tickers in the 1982-1987 window: {len(tickers_1982_1987)}")

# We verify here that they are on NYSE/AMEX

# Checking for 10 consecutive quarterly announcment for these shortlisted firms, since the beginning of Compustat
dfQ = dfQ.sort_values(by=["tic", "fyearq", "fqtr"])
dfQ["pidx"] = dfQ["fyearq"] * 4 + dfQ["fqtr"]

df_filtered = dfQ[dfQ["tic"].isin(tickers_1982_1987)].copy()

sub = df_filtered[df_filtered["epspxq"].notna()].copy()
sub["break"] = sub.groupby("tic")["pidx"].diff().ne(1)
sub["block"] = sub.groupby("tic")["break"].cumsum()

# 10 or more consecutive quarters
sizes = sub.groupby(["tic", "block"]).size().reset_index(name="count")
valid_firms = sizes[sizes["count"] >= 10]["tic"].nunique()
print(f"{valid_firms} shortlisted firms (10-consecutive quarters, ticker present in 1982-1987)")


Number of firms in the 1982-1987 window: 11004
7695 shortlisted firms (10-consecutive quarters, ticker present in 1982-1987)


We then load CRSP Daily files, filtering firms not listed on NYSE/AMEX, following Bernard & Thomas method.

In [ ]:
# Loading data
dfD = pd.read_csv("data/CRSP_Daily_1974_1986.csv")
dfD_oos = pd.read_csv("data/CRSP_Daily_2012_2025.csv")

# We only select NYSE/AMEX listed firms
dfD = dfD[dfD['PrimaryExch'].isin(['A', 'N'])]
dfD_oos = dfD_oos[dfD_oos['PrimaryExch'].isin(['A', 'N'])]

print(f"CRSD Daily observations (1974-1986): {len(dfD)}\n   {len(dfD['PERMNO'].unique())} firms (PERMCO)")
print(f"CRSD Daily observations (2012-2024): {len(dfD_oos)}\n   {len(dfD_oos['PERMNO'].unique())} firms (PERMCO)")
dfD.head()

CRSD Daily observations (1974-1986): 8036197
   3959  (PERMCO)
CRSD Daily observations (2012-2024): 9193068
   4939 firms (PERMCO)


,PERMNO,HdrCUSIP,PrimaryExch,SecurityNm,SecurityType,Ticker,PERMCO,DlyCalDt,DlyPrc,DlyCap,DlyClose,DlyLow,DlyHigh,DlyOpen
1238,10006,00080010,N,A C F INDUSTRIES INC; COM NONE; CONS,EQTY,ACF,22156,1974-01-02,57.875,324447.25,57.875,57.50,58.50,NaN
1239,10006,00080010,N,A C F INDUSTRIES INC; COM NONE; CONS,EQTY,ACF,22156,1974-01-03,61.250,343367.50,61.250,59.50,61.25,NaN
1240,10006,00080010,N,A C F INDUSTRIES INC; COM NONE; CONS,EQTY,ACF,22156,1974-01-04,59.750,334958.50,59.750,59.75,61.25,NaN
1241,10006,00080010,N,A C F INDUSTRIES INC; COM NONE; CONS,EQTY,ACF,22156,1974-01-07,58.250,326549.50,58.250,57.25,59.50,NaN
1242,10006,00080010,N,A C F INDUSTRIES INC; COM NONE; CONS,EQTY,ACF,22156,1974-01-08,56.875,318841.25,56.875,56.25,58.00,NaN


In [40]:
# We match valid Compustat firms with valid CRSP firms based on PERMCO matching.